# Training Pipeline - ChatKasir

- Nama: Achmad Rif'an (AI-1 Model Architect)
- Minggu: 2 - Pengembangan Fitur Inti (27 April - 1 Mei)


## 1. Setup Google Colab dengan GPU

In [1]:
import tensorflow as tf
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split

# verifikasi versi
print(f"TensorFlow : {tf.__version__}")
print(f"NumPy      : {np.__version__}")
print(f"Pandas     : {pd.__version__}")

TensorFlow : 2.20.0
NumPy      : 2.0.2
Pandas     : 2.2.2


In [2]:
# verifikasi GPU
gpus = tf.config.list_physical_devices('GPU')
print(f"\nGPU tersedia: {len(gpus) > 0}")

if gpus:
    # tampilkan detail GPU yang aktif agar terdokumentasi
    for gpu in gpus:
        print(f"Nama GPU    : {gpu.name}")

    # aktifkan memory growth - GPU tidak langsung mengambil semua VRAM
    # tapi mengalokasikan secara bertahap sesuai kebutuhan
    # ini mencegah crash "out of memory" di awal training
    for gpu in gpus:
        tf.config.experimental.set_memory_growth(gpu, True)
    print("Memory growth: aktif")
else:
    print("PERINGATAN: GPU tidak aktif! Cek Runtime -> Change runtime type")


GPU tersedia: False
PERINGATAN: GPU tidak aktif! Cek Runtime -> Change runtime type


## 2. Load Dataset

In [3]:
# load dataset food
url_food_utama = "https://drive.google.com/uc?id=1xpoFjqAT9K0uwzSpVADm_EfKqG7dxVUI"

df_food = pd.read_csv(url_food_utama)
df_food.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 18558 entries, 0 to 18557
Data columns (total 1 columns):
 #   Column  Non-Null Count  Dtype 
---  ------  --------------  ----- 
 0   name    18558 non-null  object
dtypes: object(1)
memory usage: 145.1+ KB


In [4]:
# load dataset slang
url_slang_utama = "https://drive.google.com/uc?id=1G14C1qcqOp06Xs1HFiorE3Us_LLtaBs7"

df_slang = pd.read_csv(url_slang_utama)
df_slang.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1231 entries, 0 to 1230
Data columns (total 2 columns):
 #   Column  Non-Null Count  Dtype 
---  ------  --------------  ----- 
 0   slang   1231 non-null   object
 1   formal  1231 non-null   object
dtypes: object(2)
memory usage: 19.4+ KB


In [5]:
# load dataset sintetis
url_synthetic_10000 = "https://drive.google.com/uc?id=15luIrYGJEZpbH-Wf7foXHXo3LBt0MBqU"

df_synthetic = pd.read_csv(url_synthetic_10000)
df_synthetic.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10050 entries, 0 to 10049
Data columns (total 5 columns):
 #   Column        Non-Null Count  Dtype 
---  ------        --------------  ----- 
 0   input_text    10050 non-null  object
 1   product       10050 non-null  object
 2   quantity      10050 non-null  int64 
 3   price_satuan  10050 non-null  int64 
 4   pattern       10050 non-null  int64 
dtypes: int64(3), object(2)
memory usage: 392.7+ KB


## 3. Pipeline Data Loading
Ada 6 tahap yang dilakukan:

1. Memuat dan memvalidasi dataset
2. Tokenisasi: mengubah teks percakapan menjadi array integer
3. Encoding label: setiap nama produk dipetakan ke integer
4. Normalisasi price dan persiapan array
5. Split dataset menjadi training set dan validation set
6. Membangun tf.data.Dataset untuk mengirimkan data ke model saat training


### Tahap 1: Validasi Dataset Sintetis

In [6]:
print(f"Shape dataset   : {df_synthetic.shape}")
print(f"Kolom yang ada  : {list(df_synthetic.columns)}")


Shape dataset   : (10050, 5)
Kolom yang ada  : ['input_text', 'product', 'quantity', 'price_satuan', 'pattern']


In [7]:
# validasi nama kolom sesuai kesepakatan
kolom_wajib = ["input_text", "product", "quantity", "price_satuan", "pattern"]

kolom_hilang = [k for k in kolom_wajib if k not in df_synthetic.columns]
if kolom_hilang:
    print(f"ERROR: Kolom berikut tidak ditemukan: {kolom_hilang}")
    print("Minta Faradi (DS-1) untuk menyesuaikan nama kolom!")
else:
    print("Semua kolom sesuai")

Semua kolom sesuai


In [8]:
# tampilkan sampel data untuk inspeksi visual
print(f"Sampel 3 baris pertama:")
print(df_synthetic[["input_text", "product", "quantity", "price_satuan"]].head(3).to_string())

Sampel 3 baris pertama:
                                                                                                                  input_text                        product  quantity  price_satuan
0  minta 7 nasi kari limasari nasi putih ya [SEP] oke kak nasi kari limasari nasi putih harganya 22 ribu totalnya rp 154.000  nasi kari limasari nasi putih         7         22000
1                         bu mau pesen 4 mie goreng djawa [SEP] noted kak mie goreng djawa rp 48.000 per porsi totalnya 192k               mie goreng djawa         4         48000
2                                                                         4 teh obenk dong [SEP] oke kak pesanannya masuk ya                      teh obenk         4            -1


In [9]:
# cek distribusi pattern, memastikan proporsi seimbang
print(f"Distribusi pattern:")

for pattern, count in df_synthetic['pattern'].value_counts().sort_index().items():
    pct = count / len(df_synthetic) * 100
    print(f"Pola {pattern}: {count} baris ({pct:.1f}%)")

Distribusi pattern:
Pola 1: 3922 baris (39.0%)
Pola 2: 3888 baris (38.7%)
Pola 3: 2240 baris (22.3%)


In [10]:
# cek baris dengan harga satuan -1
n_null = (df_synthetic['price_satuan'] == -1).sum()

print(f"Baris tanpa harga satuan (-1): {n_null} ({n_null/len(df_synthetic):.1%})")

Baris tanpa harga satuan (-1): 1750 (17.4%)


In [11]:
# cek apakah ada nilai yang null (NaN)
print(f"Nilai NaN per kolom:")
print(df_synthetic[kolom_wajib].isnull().sum().to_string())

Nilai NaN per kolom:
input_text      0
product         0
quantity        0
price_satuan    0
pattern         0


### Tahap 2: Tokenisasi Teks
Mengubah teks chat obrolan pesanan pembeli dan penjual menjadi array integer menggunakan TextVectorization

In [12]:
# hyperparameter
# diambil dari 01_model_architecture.ipynb Minggu 1
VOCAB_SIZE = 10000
MAX_SEQ_LEN = 30

# TextVectorization
vectorizer = tf.keras.layers.TextVectorization(
    max_tokens=VOCAB_SIZE, # maksimum kata unik dalam kamus
    output_sequence_length=MAX_SEQ_LEN, # panjang urutan maksmimum
    output_mode="int", # output berupa integer (bukan one-hot)
    standardize="lower_and_strip_punctuation", # lowercase dan hapus tanda baca
    name="text_vectorizer"
)

# fitting vectorizer, membangun kamus kata dari seluruh input_text
# dilakukan sekali pada training set, bukan test set
# untuk menghindari data leakage dari test set ke training
print("Membangun kamus kata dari dataset...")
vectorizer.adapt(df_synthetic["input_text"].values)

Membangun kamus kata dari dataset...


In [13]:
# verifikasi kamus yang terbentuk
vocab = vectorizer.get_vocabulary()

print(f"Kamus terbentuk dengan {len(vocab)} kata unik")
print(f"10 kata pertama (termasuk token khusus): {vocab[:5]}")
print(f"10 kata paling umum: {vocab[2:12]}")

# catatan: vocab[0] = "" (padding), vocab[1] = "[UNK]" (unknown)
# kata-kata selanjutnya diurutkan dari yang paling sering muncul

Kamus terbentuk dengan 4056 kata unik
10 kata pertama (termasuk token khusus): ['', '[UNK]', np.str_('kak'), np.str_('sep'), np.str_('ya')]
10 kata paling umum: [np.str_('kak'), np.str_('sep'), np.str_('ya'), np.str_('totalnya'), np.str_('nasi'), np.str_('ayam'), np.str_('oke'), np.str_('rp'), np.str_('dong'), np.str_('siap')]


In [14]:
# uji coba tokenisasi pada 1 contoh teks
contoh_teks = df_synthetic["input_text"].iloc[1]
contoh_token = vectorizer([contoh_teks]).numpy()[0]

print(f"Contoh tokenisasi:")
print(f"Teks asli: '{contoh_teks}'")
print(f"Hasil token: {contoh_token}")

Contoh tokenisasi:
Teks asli: 'bu mau pesen 4 mie goreng djawa [SEP] noted kak mie goreng djawa rp 48.000 per porsi totalnya 192k'
Hasil token: [  66   13   27   38   24   17 1549    3   21    2   24   17 1549    9
  192   15   16    5 1603    0    0    0    0    0    0    0    0    0
    0    0]


### Tahap 3: Encoding Label Product
Mengubah nama produk dari teks menjadi integer

In [15]:
# StringLookup
# layer tensorflow yang memetakan string menjadi integer
# membuat kamus produk, tiap nama produk unik mendapat nomor urut yang konsisten
label_encoder_product = tf.keras.layers.StringLookup(
    output_mode="int",
    name="product_label_encoder"
)

# fit label_encoder_product dengan seluruh nama produk unik di dataset
label_encoder_product.adapt(df_synthetic["product"].values)

In [16]:
# jumlah kelas produk harus konsisten dengan NUM_PRODUCTS
# di arsitektur model 01_model_architecture.ipynb (+1 UNK)
NUM_PRODUCTS = label_encoder_product.vocabulary_size()

print(f"Jumlah kelas produk (NUM_PRODUCTS): {NUM_PRODUCTS}")
print(f"Catatan: angka ini harus sama dengan yang digunakan di build_model()")

Jumlah kelas produk (NUM_PRODUCTS): 4976
Catatan: angka ini harus sama dengan yang digunakan di build_model()


In [17]:
# uji coba encoding
contoh_produk = df_synthetic["product"].iloc[1]
contoh_encoded = label_encoder_product([contoh_produk]).numpy()[0]

print(f"\nContoh label encoding:")
print(f"  Nama produk : '{contoh_produk}'")
print(f"  Hasil encode: {contoh_encoded}")


Contoh label encoding:
  Nama produk : 'mie goreng djawa'
  Hasil encode: 912


In [18]:
# simpan vocabulary label encoder untuk digunakan saat inferensi
# dibutuhkan untuk mengubah integer kembali ke nama produk
# (de-encoding) saat model sudah terlatih
produk_vocab = label_encoder_product.get_vocabulary()

print(f"5 produk pertama dalam kamus: {produk_vocab[:5]}")

# produk_vocab[0] = "[UNK]" untuk produk yang tidak dikenal
# produk_vocab[1:] = nama-nama produk yang dikenal

5 produk pertama dalam kamus: ['[UNK]', np.str_('nasi telor kalimantan'), np.str_('es kopi tem'), np.str_('nasi nila saos tiram'), np.str_('ice kopi taro')]


### Tahap 4: Normalisasi Price dan Persiapan Array

In [19]:
# tokenisasi seluruh input_text sekaligus
X = vectorizer(df_synthetic["input_text"].values).numpy()

print(f"Shape X (input model): {X.shape}")

Shape X (input model): (10050, 30)


In [20]:
# label encoding untuk product
y_product = label_encoder_product(df_synthetic["product"].values).numpy()

print(f"Shape y_product: {y_product.shape}")

Shape y_product: (10050,)


In [21]:
# quantity langsung diambil sebagai integer
y_quantity = df_synthetic["quantity"].values.astype(np.float32)

print(f"Shape y_quantity: {y_quantity.shape}")
print(f"Range quantity: {y_quantity.min():.0f} - {y_quantity.max():.0f}")

Shape y_quantity: (10050,)
Range quantity: 1 - 10


In [22]:
# price: normalisasi dengan membagi 1000
# kecuali baris dengan sentinel -1 yang harus dibiarkan -1
y_price_raw = df_synthetic["price_satuan"].values.astype(np.float32)
y_price = np.where(
    y_price_raw == -1,       # kondisi: kalau nilainya sentinel -1
    -1.0,                    # jika benar: biarkan -1 (jangan dinormalisasi)
    y_price_raw / 1000.0     # jika salah: normalisasi dengan bagi 1000
)

print(f"Shape y_price             : {y_price.shape}")
print(f"Range price valid (÷1000) : {y_price[y_price != -1].min():.1f} – {y_price[y_price != -1].max():.1f}")
print(f"Jumlah sentinel -1        : {(y_price == -1).sum()}")

Shape y_price             : (10050,)
Range price valid (÷1000) : 3.0 – 75.0
Jumlah sentinel -1        : 1750


### Tahap 5: Split Train & Validation
Membagi dataset menjadi training set sebesar 80% dan validation set 20%

In [23]:
X_train, X_val, y_product_train, y_product_val, y_quantity_train, y_quantity_val, y_price_train, y_price_val = train_test_split(
    X, y_product, y_quantity, y_price,
    test_size=0.2,
    random_state=42, # memastikan split yang sama setiap kali dijalankan
    stratify=df_synthetic["pattern"].values # menjaga proporsi ketiga pola tetap seimbang
)

print(f"Training set   : {len(X_train)} baris ({len(X_train)/len(X):.0%})")
print(f"Validation set : {len(X_val)} baris ({len(X_val)/len(X):.0%})")

print(f"\nVerifikasi shape setelah split:")
print(f"X_train        : {X_train.shape}")
print(f"y_product_train: {y_product_train.shape}")
print(f"y_price_train  : {y_price_train.shape}")

Training set   : 8040 baris (80%)
Validation set : 2010 baris (20%)

Verifikasi shape setelah split:
X_train        : (8040, 30)
y_product_train: (8040,)
y_price_train  : (8040,)


### Tahap 6: Membangun tf.data.Dataset

Membungkus array numpy menjadi tf.data.Dataset yang siap digunakan untuk training atau validasi.

Parameter shuffle=True digunakan untuk training set agar urutan data acak setiap epoch. Mencegah model belajar pola berdasarkan urutan.

Untuk validation set, shuffle=False karena tidak diperlukan.

In [27]:
# model memproses 32 kalimat sekaligus dalam 1 langkah, bukan satu per satu
BATCH_SIZE = 32

def buat_dataset(X, y_product, y_quantity, y_price, shuffle=False):

    # dataset input: dictionary karena model.fit() mengharapkan format ini
    # saat model memiliki multiple inputs (meskipun di sini hanya satu input)
    dataset = tf.data.Dataset.from_tensor_slices((
        {"input_tokens": X}, # input ke model
        {   # label (target) untuk setiap output head
            "product": y_product,
            "quantity": y_quantity,
            "price_satuan": y_price
        }
    ))

    if shuffle:
        # buffer size = jumlah data
        dataset = dataset.shuffle(buffer_size=len(X), seed=42)

    dataset = dataset.batch(BATCH_SIZE)

    # prefetch(tf.data.AUTOTUNE): saat GPU sedang melatih batch ke-N,
    # CPU sudah mempersiapkan batch ke-N+1 di background
    # ini menghilangkan "waktu menunggu" antara batch dan mempercepat training
    dataset = dataset.prefetch(tf.data.AUTOTUNE)

    return dataset

train_dataset = buat_dataset(
    X_train, y_product_train, y_quantity_train, y_price_train, shuffle=True
)

val_dataset = buat_dataset(
    X_val, y_product_val, y_quantity_val, y_price_val, shuffle=False
)

# hitung jumlah batch
n_train_batches = len(X_train) // BATCH_SIZE
n_val_batches   = len(X_val) // BATCH_SIZE

print(f"Training dataset   : {n_train_batches} batch x {BATCH_SIZE} sampel")
print(f"Validation dataset : {n_val_batches} batch x {BATCH_SIZE} sampel")

Training dataset   : 251 batch x 32 sampel
Validation dataset : 62 batch x 32 sampel


In [28]:
# verifikasi satu batch untuk memastikan format sudah benar
print(f"Verifikasi format satu batch:")

for inputs_batch, labels_batch in train_dataset.take(1):
    print(f"Input 'input_tokens' shape : {inputs_batch['input_tokens'].shape}")
    print(f"Label 'product' shape      : {labels_batch['product'].shape}")
    print(f"Label 'quantity' shape     : {labels_batch['quantity'].shape}")
    print(f"Label 'price_satuan' shape : {labels_batch['price_satuan'].shape}")

Verifikasi format satu batch:
Input 'input_tokens' shape : (32, 30)
Label 'product' shape      : (32,)
Label 'quantity' shape     : (32,)
Label 'price_satuan' shape : (32,)
